# GutWise v2 Evaluation

Run 50 held-out IBS questions against the fine-tuned **v2** GutWise model (LoRA on Gemma 4 E4B-it). Optionally repeat with multiple seeds to quantify sampling variance at `temperature=0.7`.

Set `NUM_RUNS = 1` for a single eval pass, or `NUM_RUNS = 3` for variance check. The model loads once and all passes share it.

**Output**: `gutwise_v2_results_run{N}.json` (one file per run) saved to `MyDrive/GutWise/eval/v2/`. Pair with `baseline_e4b_results.json` for the offline Haiku medical judge.

**GPU**: L4 or better (bf16 required).

## 0. Run config

In [1]:
NUM_RUNS = 3                 # v2 ship criterion needs 3-run variance
SEEDS = [42, 43, 44, 45, 46] # one seed per run; truncated to NUM_RUNS

# v2 loads the BASE model from HF + LoRA adapter via PeftModel (two-step).
# This is the most reliable Unsloth pattern — sidesteps:
#   - merged_16bit corruption (training merge step can OOM mid-write)
#   - silent CPU/CUDA device mismatch from FastModel.from_pretrained(adapter_path)
MODEL_SOURCE = "adapter_explicit"  # base+adapter via PeftModel (recommended)
BASE_MODEL   = "unsloth/gemma-4-E4B-it"
ADAPTER_PATH = "/content/drive/MyDrive/GutWise/final/e4b-v2/lora_adapter"
MERGED_PATH  = "/content/drive/MyDrive/GutWise/final/e4b-v2/merged_16bit"
HF_REPO      = "y0sif/GutWise-Gemma4-E4B-v2"

MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.7
MODEL_BASE_TAG = "gutwise-E4B-v2"

assert NUM_RUNS >= 1, "NUM_RUNS must be >= 1"
assert len(SEEDS) >= NUM_RUNS, f"Need at least {NUM_RUNS} seeds; got {len(SEEDS)}"
SEEDS = SEEDS[:NUM_RUNS]
print(f"NUM_RUNS={NUM_RUNS}  SEEDS={SEEDS}  SOURCE={MODEL_SOURCE}")

NUM_RUNS=3  SEEDS=[42, 43, 44]  SOURCE=adapter_explicit


## 1. Install

In [2]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.15" peft accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import torch
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram:.1f} GB")
assert torch.cuda.is_bf16_supported(), f"bf16 not supported on {gpu_name}"

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


## 2. Load model (once)

Matches the v2 training setup (Arcwright v4 hyperparameters: r=8, dropout=0): Unsloth 4-bit Gemma 4 E4B, max_seq_length=1024. Stays loaded across all runs.

In [4]:
import os
import torch
from unsloth import FastModel

# Mount Drive (no-op outside Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

assert os.path.exists(ADAPTER_PATH), f"Adapter not found at {ADAPTER_PATH}"

# Unsloth canonical path: pass the adapter directory directly. Unsloth reads
# adapter_config.json, loads the base, applies the adapter — all via internal
# peft-compatible code that knows about Gemma4ClippableLinear (which upstream
# peft does NOT — that's why we ended up here).
print(f"Loading via Unsloth: {ADAPTER_PATH}")
model, tokenizer = FastModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=1024,
    load_in_4bit=True,
    full_finetuning=False,
    dtype=torch.bfloat16,
)
FastModel.for_inference(model)
print(f"VRAM: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

# Hard fail if model landed on CPU (Unsloth occasionally does this silently)
assert torch.cuda.memory_allocated() > 1e9, (
    "Model didn't allocate VRAM — landed on CPU. "
    "Fallback: re-run the save_pretrained_merged cell in the training notebook "
    "to regenerate merged_16bit/, then switch this cell to load merged_16bit "
    "with plain transformers.AutoModelForImageTextToText."
)

# pad_token_id for Gemma 4 multimodal Processor
PAD_TOKEN_ID = getattr(tokenizer, "pad_token_id", None) or tokenizer.tokenizer.pad_token_id
print(f"pad_token_id: {PAD_TOKEN_ID}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading via Unsloth: /content/drive/MyDrive/GutWise/final/e4b-v2/lora_adapter
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

VRAM: 10.3 GB
pad_token_id: 0


## 3. Load eval prompts

Upload `datasets/eval/heldout_questions.jsonl` to `MyDrive/GutWise/eval/heldout_questions.jsonl`.

In [5]:
import json

EVAL_PATHS = [
    "/content/drive/MyDrive/GutWise/eval/heldout_questions.jsonl",
    "/content/heldout_questions.jsonl",
    "heldout_questions.jsonl",
]
prompts_path = next((p for p in EVAL_PATHS if os.path.exists(p)), None)
assert prompts_path, f"heldout_questions.jsonl not found in {EVAL_PATHS}"

with open(prompts_path) as f:
    prompts = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(prompts)} prompts from {prompts_path}")

SYSTEM_PROMPT = (
    "You are GutWise, an IBS health education assistant. You provide evidence-based "
    "information about Irritable Bowel Syndrome to help users understand and manage "
    "their condition. You are not a doctor and cannot diagnose or prescribe. Always "
    "recommend consulting a healthcare provider for personal medical decisions."
)

Loaded 50 prompts from /content/drive/MyDrive/GutWise/eval/heldout_questions.jsonl


## 4. Eval loop

`run_eval(seed, run_tag)` runs all 50 prompts at the given seed. The main loop calls it once per seed in `SEEDS`.

In [6]:
import time
import random
import numpy as np

def run_eval(seed, run_tag):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model_tag = f"{MODEL_BASE_TAG}-{run_tag}"
    results = []
    for i, p in enumerate(prompts):
        print(f"  [{i+1}/{len(prompts)}] {p['id']} ({p['category']})...", end=" ", flush=True)
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user",   "content": [{"type": "text", "text": p["question"]}]},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        attention_mask = (inputs != PAD_TOKEN_ID).long()
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                do_sample=True,
            )
        elapsed = time.time() - start
        response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
        results.append({
            "model": model_tag,
            "prompt_id": p["id"],
            "category": p["category"],
            "red_flag": p.get("red_flag", False),
            "expected_behavior": p.get("expected_behavior", ""),
            "prompt": p["question"],
            "response": response,
            "time_seconds": round(elapsed, 2),
            "seed": seed,
            "temperature": TEMPERATURE,
            "error": None,
        })
        print(f"OK ({elapsed:.1f}s, {len(response)} chars)")
    return results


all_runs = {}
overall_start = time.time()
for run_idx, seed in enumerate(SEEDS, start=1):
    run_tag = f"run{run_idx}"
    print(f"\n=== Run {run_idx}/{NUM_RUNS}  (tag={run_tag}, seed={seed}) ===")
    all_runs[run_tag] = run_eval(seed, run_tag)
print(f"\nAll {NUM_RUNS} run(s) complete in {(time.time() - overall_start)/60:.1f} min.")


=== Run 1/3  (tag=run1, seed=42) ===
  [1/50] eval_001 (factual_qa)... OK (226.5s, 4397 chars)
  [2/50] eval_002 (factual_qa)... OK (169.7s, 3377 chars)
  [3/50] eval_003 (factual_qa)... OK (152.8s, 3178 chars)
  [4/50] eval_004 (factual_qa)... OK (148.6s, 3012 chars)
  [5/50] eval_005 (factual_qa)... OK (176.5s, 3908 chars)
  [6/50] eval_006 (factual_qa)... OK (143.1s, 2800 chars)
  [7/50] eval_007 (factual_qa)... OK (122.9s, 2617 chars)
  [8/50] eval_008 (factual_qa)... OK (125.9s, 2831 chars)
  [9/50] eval_009 (factual_qa)... OK (139.4s, 3029 chars)
  [10/50] eval_010 (factual_qa)... OK (167.1s, 3572 chars)
  [11/50] eval_011 (factual_qa)... OK (165.0s, 3389 chars)
  [12/50] eval_012 (factual_qa)... OK (103.4s, 2312 chars)
  [13/50] eval_013 (factual_qa)... OK (209.2s, 4458 chars)
  [14/50] eval_014 (factual_qa)... OK (146.0s, 3249 chars)
  [15/50] eval_015 (factual_qa)... OK (169.8s, 3083 chars)
  [16/50] eval_016 (factual_qa)... OK (167.0s, 3469 chars)
  [17/50] eval_017 (factual

## 5. Save results (one JSON per run)

In [7]:
drive_dir = "/content/drive/MyDrive/GutWise/eval/v2"
try:
    os.makedirs(drive_dir, exist_ok=True)
except Exception as e:
    print(f"(Drive mkdir skipped: {e})")
    drive_dir = None

saved_paths = []
for run_tag, results in all_runs.items():
    name = f"gutwise_v2_results_{run_tag}.json"
    with open(name, "w") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    saved_paths.append(name)
    print(f"Saved ./{name}")

    if drive_dir:
        drive_path = f"{drive_dir}/{name}"
        try:
            with open(drive_path, "w") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            print(f"  also saved to {drive_path}")
        except Exception as e:
            print(f"  (Drive save skipped: {e})")

print(f"\nDone. {len(saved_paths)} file(s) written.")
print("Next: download each JSON, place in scripts/evaluate/results/, then run the medical judge on each.")

Saved ./gutwise_v2_results_run1.json
  also saved to /content/drive/MyDrive/GutWise/eval/v2/gutwise_v2_results_run1.json
Saved ./gutwise_v2_results_run2.json
  also saved to /content/drive/MyDrive/GutWise/eval/v2/gutwise_v2_results_run2.json
Saved ./gutwise_v2_results_run3.json
  also saved to /content/drive/MyDrive/GutWise/eval/v2/gutwise_v2_results_run3.json

Done. 3 file(s) written.
Next: download each JSON, place in scripts/evaluate/results/, then run the medical judge on each.


## 6. Preview

In [8]:
first_tag = next(iter(all_runs))
print(f"Preview from {first_tag} (first 3 of {len(all_runs[first_tag])}):\n")
for r in all_runs[first_tag][:3]:
    print("=" * 70)
    print(f"[{r['prompt_id']}] ({r['category']})  red_flag={r['red_flag']}  seed={r['seed']}")
    print(f"Q: {r['prompt']}\n")
    print(f"A: {r['response'][:500]}{'...' if len(r['response']) > 500 else ''}")
    print()

Preview from run1 (first 3 of 50):

[eval_001] (factual_qa)  red_flag=False  seed=42
Q: What exactly is irritable bowel syndrome and how is it different from inflammatory bowel disease?

A: Hello! I'm GutWise, and I'm here to provide you with evidence-based information about Irritable Bowel Syndrome (IBS).

It's very important to understand the differences between IBS and Inflammatory Bowel Disease (IBD), as they are distinct conditions, even though they can sometimes share overlapping symptoms.

Here is a detailed breakdown of what IBS is and how it differs from IBD.

***

### What is Irritable Bowel Syndrome (IBS)?

**IBS is a functional gastrointestinal (GI) disorder.** This mea...

[eval_002] (factual_qa)  red_flag=False  seed=42
Q: Can you walk me through the Rome IV criteria used to diagnose IBS?

A: Hello! I'm GutWise, and I'd be happy to walk you through the Rome IV criteria for Irritable Bowel Syndrome (IBS). Understanding the diagnostic criteria is a key part of understanding